In [ ]:
import tensorflow as tf
import numpy as np
from utils import ASSETS_DIR
from tensorflow.keras import layers, models, regularizers
from scikeras.wrappers import KerasClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from tensorflow.keras.callbacks import EarlyStopping
from MLPipeline import *
from tensorflow.keras.callbacks import Callback
from tensorflow.keras import backend as K
import gc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import RandomizedSearchCV


In [ ]:

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Applica il memory growth a tutte le GPU rilevate
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ Allocazione dinamica della memoria GPU attivata.")
    except RuntimeError as e:
        # Questo errore capita se TensorFlow ha già inizializzato la GPU
        print(e)

In [ ]:
def build_tunable_autoencoder(input_dim=128, l1_reg=1e-4, l2_reg=0.001, drop_rate=0.3, **kwargs):
    inputs = layers.Input(shape=(input_dim,))

    

    attention_weights = layers.Dense(input_dim, activation="sigmoid",
                                    name="feature_gating",
                                    kernel_initializer="glorot_uniform",              # Protegge dal Vanishing Gradient
                                    bias_initializer=tf.keras.initializers.Constant(0.5),)(inputs)
    
    # Moltiplica ogni input originale per il suo peso specifico (Element-wise)
    gated_inputs = layers.Multiply()([inputs, attention_weights])

    # ENCODER (GELU + He Normal)
    encoder = layers.Dense(64, activation="gelu", kernel_initializer="he_normal", kernel_regularizer=regularizers.l2(l2_reg))(gated_inputs)
    encoder = layers.BatchNormalization()(encoder)
    encoder = layers.Dropout(drop_rate)(encoder)

    encoder = layers.Dense(32, activation="gelu", kernel_initializer="he_normal", kernel_regularizer=regularizers.l2(l2_reg))(encoder)
    encoder = layers.BatchNormalization()(encoder)
    # Riduciamo leggermente il dropout più andiamo in profondità
    encoder = layers.Dropout(max(0.0, drop_rate - 0.1))(encoder)

    # BOTTLENECK (RELU + L1 parametrizzata)
    sparse_bottleneck = layers.Dense(16, activation="relu", activity_regularizer=regularizers.l1(l1_reg))(encoder)

    # CLASSIFICATORE
    classifier = layers.Dense(32, activation="gelu", kernel_initializer="he_normal", kernel_regularizer=regularizers.l2(l2_reg))(sparse_bottleneck)
    classifier = layers.BatchNormalization()(classifier)
    classifier = layers.Dropout(max(0.0, drop_rate - 0.1))(classifier)

    classifier = layers.Dense(64, activation="gelu", kernel_initializer="he_normal", kernel_regularizer=regularizers.l2(l2_reg))(sparse_bottleneck)
    classifier = layers.BatchNormalization()(classifier)
    classifier = layers.Dropout(drop_rate)(classifier)


    output = layers.Dense(3, activation='softmax')(classifier)

    model = models.Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate= 0.001),
        loss='sparse_categorical_crossentropy',
        metrics=["accuracy"]
    )

    return model


class ClearMemoryCallback(Callback):
    def on_train_end(self, logs=None):
        """
        Questo metodo scatta in automatico nell'istante in cui 
        l'EarlyStopping ferma l'addestramento del fold.
        """
        gc.collect()          # Forza Python a svuotare la RAM di sistema
        K.clear_session()     # Distrugge il grafo di TensorFlow, liberando la GPU


In [ ]:
memory_cleaner = ClearMemoryCallback()

df = pd.read_parquet(ASSETS_DIR / 'final_df.parquet')



X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

In [ ]:

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")

# 3. CONFIGURAZIONE DEL KNN IMPUTER

imputer = KNNImputer(n_neighbors=7, weights='distance')

# 4. ADDESTRAMENTO E TRASFORMAZIONE
# Il modello "impara" le distribuzioni SOLO da X_train
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Il modello applica quanto imparato su X_test (senza barare)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 5. ARROTONDAMENTO PER DATI CLINICI DISCRETI
# Riportiamo le medie del KNN a numeri interi (es. 0.66 diventa 1.0)
X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")



early_stopping = EarlyStopping(
    monitor='val_loss',         # Guarda l'errore sul set di validazione interno
    patience=10,                # Aspetta 10 epoche senza miglioramenti prima di arrendersi
    restore_best_weights=True   # IL SEGRETO: Riporta indietro la rete alla sua forma perfetta
)



In [ ]:

param_grid_tf = {
    'classifier__model__l1_reg': [1e-4, 1e-5],    # Testiamo due livelli di sparsità
    'classifier__model__l2_reg': [0.01, 0.05, 0.001],   # Testiamo la forza della regolarizzazione
    'classifier__model__drop_rate': [0.2, 0.3, 0.4],   # Testiamo quanto "spegnere" i neuroni
    'classifier__batch_size': [64]            # Questo è un parametro nativo di KerasClassifier
}

with tf.device('/GPU:0'):
    
    keras_mlp = KerasClassifier(
        model=build_tunable_autoencoder,
        epochs=100, # Iniziamo con 50 per un test rapido
        batch_size=128,
        verbose=0, 
        model__l1_reg=1e-4,
        model__l2_reg=0.001,
        model__drop_rate=0.3,
        random_state=42,
        validation_split=0.15,
        callbacks=[early_stopping, memory_cleaner],

    )

    pipeline_tf = ImbPipeline(steps=[
        ('scaler', MinMaxScaler()),
        ('smoteenn', SMOTEENN(random_state=42)),
        ('classifier', keras_mlp)
    ])

''' print("Avvio K-Fold Cross Validation con Rete Neurale (Auto-tuning delle epoche attivato)...")

# 4. Esecuzione tramite la TUA funzione
# Usa i dati X_iter e y_iter che abbiamo pulito prima col taglio basato sulla gravità
# La funzione calcolerà e stamperà tutte le tue metriche personalizzate
y_true_bin, y_proba_tf = train_model_evaluate(
    X=X_train_imp, 
    y=y_train, 
    model=pipeline_tf, 
    use_cv=True,
    cv_strategy=cv_strategy,
)
print("Test completato!")'''




In [ ]:
print("Avvio RandomizedSearchCV (Test di 5 combinazioni). RAM sotto controllo...")

# 4. IL MOTORE DI RICERCA CASUALE
random_search_tf = RandomizedSearchCV(
    estimator=pipeline_tf,
    param_distributions=param_grid_tf,
    n_iter=8,             # IL TRUCCO: Prova solo 5 combinazioni
    cv=3,                 # 3 split di test (Totale: 15 addestramenti invece di 48)
    scoring='f1_macro',   
    n_jobs=1,             # SEMPRE a 1 per proteggere la memoria
    verbose=2,            # Così vedi in diretta cosa sta testando
    random_state=42
)

# 5. Eseguiamo la ricerca sui dati di Train
random_search_tf.fit(X_train_imp, y_train)

print("\n=== RISULTATI RANDOMIZED SEARCH ===")
print(f"Miglior F1-Score (Macro) in CV: {random_search_tf.best_score_:.4f}")
print(f"Migliori Iperparametri:\n{random_search_tf.best_params_}")

In [ ]:
print("Avvio  Rete Neurale (Auto-tuning delle epoche attivato)...")
    
    # 4. Esecuzione tramite la TUA funzione
    # Usa i dati X_iter e y_iter che abbiamo pulito prima col taglio basato sulla gravità
    # La funzione calcolerà e stamperà tutte le tue metriche personalizzate
y_true_bin, y_proba_tf, y_pred = train_model_evaluate(
    X=X_train_imp, 
    y=y_train, 
    model=random_search_tf.best_estimator_, 
    use_cv=False,
    X_test=X_test_imp,
    y_test=y_test
)
print("Test completato!")


cm = confusion_matrix(y_test, y_pred)

etichette = ['Sani (0)', 'Alzheimer (1)', 'Lewy Body (2)']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=etichette, yticklabels=etichette,
            linewidths=1, linecolor='black')

plt.title("confusion amtrix", fontsize=14, pad=15)
plt.ylabel('Diagnosi Reale (Medico)', fontsize=12, fontweight='bold')
plt.xlabel(f'Previsione ({type(pipeline_tf).__name__})', fontsize=12, fontweight='bold')
plt.show()

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=etichette))

plot_reliability_diagram(y_true_bin, y_proba_tf[:, 2])